In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch


import sys, os
# handle paths
src = os.getcwd()
src = os.path.abspath(os.path.join(src, '../.'))
sys.path.append(src)


from aihwkit.linalg import AnalogMatrix
from cuda.shared import IdealPreset
from aihwkit.simulator.presets import ReRamSBPreset
from src.noise import ExperimentalNoiseModel, CompensatedExperimentalNoiseModel, SingleConductanceNoiseModel, CompensatedSingleConductanceNoiseModel 

from copy import deepcopy
from aihwkit.simulator.rpu_base import cuda

from src.utilities import interpolate
from aihwkit.inference.converter.conductance import SinglePairConductanceConverter




In [2]:
# Check GPU device
torch.cuda.set_device(0)
DEVICE = torch.device("cuda:0" if cuda.is_compiled() else "cpu")
print(f"Device: {DEVICE}")
RPU_BASE = ReRamSBPreset()
cur_dir = os.getcwd()

# Generate a matrix initialized with random values using a distribuiton of choice
DISTRIBUTIONS = ['normal', 'uniform']
DISTRIBUTION = DISTRIBUTIONS[0]

W_ideal = np.random.normal(0., 1. , (100, 100))
x = np.random.normal(0., 1., (100, 1)).astype("float32")
y = W_ideal.dot(x)

level = 5

# extract the noise data
variables = interpolate(levels=level, file_path=cur_dir + "/../data/matlab/4bit.mat", force_interpolation=True)
noise_types = variables['str']
noise = noise_types[0]

rpu_config = deepcopy(RPU_BASE)
noise_model = noise_model = ExperimentalNoiseModel(file_path = cur_dir + "/../data/matlab/4bit.mat",
                                                    type = noise,
                                                    levels = level,
                                                    force_interpolation = True,
                                                    compensation = False,
                                                    g_converter=SinglePairConductanceConverter(g_max=40.))
rpu_config.noise_model = noise_model

# Move W_ideal to analog tiles
W = AnalogMatrix(W_ideal, rpu_config=rpu_config, realistic=True, device = DEVICE)

y_hat = W.dot(x)
print(f"y_hat: {y_hat}")




Device: cuda:0



 --------------------------------------------------------------------------------------------------
TYPE: Prog
MEDIAN VALUES: tensor([-39.9088, -20.0080,   0.1587,  20.1705,  40.1831], dtype=torch.float64)
STD VALUES: tensor([3.5817, 3.4527, 0.0406, 3.5286, 3.6404], dtype=torch.float64)


RuntimeError: Invalid x_input dimensions: expected [100,*] array